**Customers**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# leitura da tabela
df_customers = spark.table("medallion.bronze.tb_customers")

# tranformação de nome e tipo da coluna
df_customers = df_customers.select(
    F.col("customer_id").cast("string").alias("id_consumidor"),
    F.col("customer_unique_id").cast("string").alias("id_consumidor_unico"),
    F.col("customer_zip_code_prefix").cast("int").alias("prefixo_cep"),
    F.col("customer_city").cast("string").alias("cidade"),
    F.col("customer_state").cast("string").alias("estado"),
    F.col("customer_name").cast("string").alias("nome_consumidor"),
    F.col("customer_gender").cast("string").alias("genero_consumidor"),
    F.col("customer_birth_date").alias("data_aniversario_consumidor"),
    F.col("customer_age").cast("int").alias("idade_consumidor"),
    "timestamp_ingestion"
)

F.to_date("data_aniversario_consumidor", "yyyy-MM-dd")

# deixando cidade e estado em maiuscula
df_customers = df_customers.withColumn("cidade", F.upper("cidade")) \
       .withColumn("estado", F.upper("estado"))

# removendo linhas em que id do consumidor é nulo
df_customers = df_customers.filter(F.col("id_consumidor").isNotNull())

# abrindo uma window function para particionar pelo id e ordenar decrescentemente pela data de ingestão
window_spec = Window.partitionBy("id_consumidor") \
    .orderBy(
        F.col("timestamp_ingestion").desc()
    )

df_customers = df_customers.withColumn(
    "row_number",
    F.row_number().over(window_spec)
)

# so deixando em que row number == 1 para não ter 2 com id igual e tirando a coluna auxiliar row number do df final
df_customers = df_customers.filter(F.col("row_number") == 1).drop("row_number")

# certificando existencia do schema silver e salvando na tabela dim_consumidores
spark.sql(f"create schema if not exists medallion.silver")
df_customers.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.dim_consumidores")

**Orders**

In [0]:
# leitura da tabela
df_orders = spark.table("medallion.bronze.tb_orders")

# renomear + cast
df_orders = df_orders.select(
    F.col("order_id").cast("string").alias("id_pedido"),
    F.col("customer_id").cast("string").alias("id_consumidor"),
    F.trim(F.lower(F.col("order_status"))).alias("status_pedido"),
    F.to_timestamp("order_purchase_timestamp", "yyyy-MM-dd HH:mm:ss").alias("data_compra"),
    F.to_timestamp("order_approved_at", "yyyy-MM-dd HH:mm:ss").alias("data_aprovacao"),
    F.to_timestamp("order_delivered_carrier_date", "yyyy-MM-dd HH:mm:ss").alias("data_envio_transportadora"),
    F.to_timestamp("order_delivered_customer_date", "yyyy-MM-dd HH:mm:ss").alias("data_entrega_cliente"),
    F.to_timestamp("order_estimated_delivery_date", "yyyy-MM-dd HH:mm:ss").alias("data_entrega_estimada"),
    "timestamp_ingestion"
)

# filtro de dados que tem que estar presentes
df_orders = df_orders.filter(
    (F.col("id_pedido").isNotNull()) &
    (F.col("id_pedido") != "") &
    (F.col("data_compra").isNotNull())
)

# tradução coluna de status
df_orders = df_orders.withColumn(
    "status_pedido",
    F.when(F.col("status_pedido") == "delivered", "entregue")
     .when(F.col("status_pedido") == "canceled", "cancelado")
     .when(F.col("status_pedido") == "shipped", "enviado")
     .when(F.col("status_pedido") == "processing", "processando")
     .when(F.col("status_pedido") == "invoiced", "faturado")
     .when(F.col("status_pedido") == "unavailable", "indisponível")
     .when(F.col("status_pedido") == "created", "criado")
     .when(F.col("status_pedido") == "approved", "aprovado")
     .otherwise("desconhecido")
)

# tempo real da entrega, calculado a partir de data_entrega_cliente e data_compra
df_orders = df_orders.withColumn(
    "tempo_entrega_dias",
    F.when(
        F.col("data_entrega_cliente").isNotNull() & F.col("data_compra").isNotNull(),
        F.datediff(F.col("data_entrega_cliente"), F.col("data_compra"))
    )
)

# remover valores negativos em tempo_entrega_dias, não faz sentido chegar antes da data que pediu
df_orders = df_orders.withColumn(
    "tempo_entrega_dias",
    F.when(F.col("tempo_entrega_dias") < 0, F.lit(None))
     .otherwise(F.col("tempo_entrega_dias"))
)

# tempo estimado da entrega, calculado a partir de data_entrega_estimada e data_compra
df_orders = df_orders.withColumn(
    "tempo_entrega_estimado_dias",
    F.when(
        F.col("data_entrega_estimada").isNotNull() & F.col("data_compra").isNotNull(),
        F.datediff(F.col("data_entrega_estimada"), F.col("data_compra"))
    )
)

# diferença entre data real e estimada da entrega
df_orders = df_orders.withColumn(
    "diferenca_entrega_dias",
    F.when(
        F.col("tempo_entrega_dias").isNotNull() &
        F.col("tempo_entrega_estimado_dias").isNotNull(),
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
    )
)

# se for diferente de entregue ou tempo_entrega_dias é nulo é não entregue, se o tempo_entrega_dias é menor que o tempo_entrega_estimado_dias ele foi entregue no prazo (logo, adiciona Sim na coluna), se não for isso é fora do prazo, (logo, adiciona Não na coluna) 
df_orders = df_orders.withColumn(
    "entrega_no_prazo",
    F.when(F.col("status_pedido") != "entregue", "Não Entregue")
     .when(F.col("tempo_entrega_dias").isNull(), "Não Entregue")
     .when(F.col("tempo_entrega_dias") <= F.col("tempo_entrega_estimado_dias"), "Sim")
     .otherwise("Não")
)

# confirma criação do schemma e salva na nova tabela
spark.sql("create schema if not exists medallion.silver")
df_orders.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.fat_pedidos")

Order Items

In [0]:
from pyspark.sql import functions as F

# leitura da tabela
df_items = spark.table("medallion.bronze.tb_order_items")

# renomeando e fazendo cast
df_items = df_items.select(
    F.col("order_id").cast("string").alias("id_pedido"),
    F.col("order_item_id").cast("int").alias("id_item"),
    F.col("product_id").cast("string").alias("id_produto"),
    F.col("seller_id").cast("string").alias("id_vendedor"),
    F.to_timestamp("shipping_limit_date").alias("data_entrega_limite"),
    F.round(F.col("price").cast("double"), 2).cast("decimal(10,2)").alias("preco_BRL"),
    F.round(F.col("freight_value").cast("double"), 2).cast("decimal(10,2)").alias("preco_frete"),
    "timestamp_ingestion"
)

# garantir valores financeiros com 2 casas decimais
df_items = df_items.withColumn(
    "preco_BRL",
    F.round(F.col("preco_BRL"), 2)
).withColumn(
    "preco_frete",
    F.round(F.col("preco_frete"), 2)
)

# remover registros inválidos que não possuem os ids necessarios para análise
df_items = df_items.filter(
    (F.col("id_pedido").isNotNull()) &
    (F.col("id_item").isNotNull())
)

# confirma criação do schemma e salva na nova tabela
spark.sql("create schema if not exists medallion.silver")
df_items.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.fat_itens_pedidos")

Order Payments

In [0]:
from pyspark.sql import functions as F

# leitura da tabela
df_payments = spark.table("medallion.bronze.tb_order_payments")

# renomeando e fazendo cast
df_payments = df_payments.select(
    F.col("order_id").cast("string").alias("id_pedido"),
    F.col("payment_sequential").cast("int").alias("n_parcela"),
    F.trim(F.lower(F.col("payment_type"))).alias("tipo_pagamento"),
    F.col("payment_installments").cast("int").alias("qtd_parcelas"),
    F.round(F.col("payment_value").cast("double"), 2).cast("decimal(10,2)").alias("valor_pagamento"),
    "timestamp_ingestion"
)

# tradução de dados da coluna para portugues
df_payments = df_payments.withColumn(
    "tipo_pagamento",
    F.when(F.col("tipo_pagamento") == "credit_card", "Cartão de Crédito")
     .when(F.col("tipo_pagamento") == "boleto", "Boleto")
     .when(F.col("tipo_pagamento") == "voucher", "Voucher")
     .when(F.col("tipo_pagamento") == "debit_card", "Cartão de Débito")
     .when(F.col("tipo_pagamento") == "not_defined", "Não Definido")
     .otherwise("Outros")
)

# remover registros inválidos que não possuem os ids necessarios para análise
df_payments = df_payments.filter(
    (F.col("id_pedido").isNotNull()) &
    (F.col("id_pedido") != "")
)

# garante existência do schema e salva na tabela final
spark.sql("create schema if not exists medallion.silver")
df_payments.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.fat_pagamentos_pedidos")

**Order Review**

In [0]:
from pyspark.sql import functions as F

# leitura da tabela
df_reviews = spark.table("medallion.bronze.tb_order_reviews")

# renomeando e tratando tipos, usando try_cast e try_to_timestamp para tolerância a falhas
df_reviews = df_reviews.select(
    F.col("review_id").cast("string").alias("id_avaliacao"),
    F.col("order_id").cast("string").alias("id_pedido"),
    F.expr("try_cast(review_score AS INT)").alias("nota_avaliacao"),  # <- aqui
    F.col("review_comment_title").cast("string").alias("titulo_comentario_avaliacao"),
    F.col("review_comment_message").cast("string").alias("comentario_avaliacao"),
    F.expr("try_to_timestamp(review_creation_date)").alias("data_criacao_avaliacao"),
    F.expr("try_to_timestamp(review_answer_timestamp)").alias("data_resposta_avaliacao"),
    F.expr("try_to_timestamp(timestamp_ingestion)").alias("timestamp_ingestion")
)

# remover registros inválidos e datas futuras
df_reviews = df_reviews.filter(
    (F.col("id_pedido").isNotNull()) & 
    (F.col("id_pedido") != "") &
    (F.col("nota_avaliacao").isNotNull()) &  # <- garante que nota seja válida
    (F.col("data_criacao_avaliacao").isNotNull()) &
    (F.col("data_criacao_avaliacao") <= F.current_timestamp())
)

# preencher comentários e títulos vazios
df_reviews = df_reviews.fillna({
    "comentario_avaliacao": "Sem comentário",
    "titulo_comentario_avaliacao": "Sem título"
}).withColumn(
    "comentario_avaliacao",
    F.when(F.col("comentario_avaliacao") == "", "Sem comentário")
     .otherwise(F.col("comentario_avaliacao"))
).withColumn(
    "titulo_comentario_avaliacao",
    F.when(F.col("titulo_comentario_avaliacao") == "", "Sem título")
     .otherwise(F.col("titulo_comentario_avaliacao"))
)

# garante timestamp_ingestion preenchido
df_reviews = df_reviews.withColumn(
    "timestamp_ingestion",
    F.coalesce(F.col("timestamp_ingestion"), F.current_timestamp())
)

# cria schema se não existir e salva na tabela final
spark.sql("create schema if not exists medallion.silver")
df_reviews.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.fat_avaliacoes_pedidos")

**Products**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# leitura da tabela
df_produtos = spark.table("medallion.bronze.tb_products")

# renomeando e tratando tipos com try_cast para tolerância a falhas
df_produtos = df_produtos.select(
    F.col("product_id").cast("string").alias("id_produto"),
    F.col("product_name").cast("string").alias("nome_produto"),
    F.col("product_category_name").cast("string").alias("categoria_produto"),
    F.col("product_name_lenght").cast("int").alias("tamanho_nome_produto"),
    F.col("product_description_lenght").cast("int").alias("tamanho_descricao_produto"),
    F.col("product_photos_qty").cast("int").alias("quantidade_fotos"),
    F.col("product_weight_g").cast("int").alias("peso_produto_gramas"),
    F.col("product_length_cm").cast("int").alias("comprimento_centimetros"),
    F.col("product_height_cm").cast("int").alias("altura_centimetros"),
    F.col("product_width_cm").cast("int").alias("largura_centimetros"),
    "timestamp_ingestion"
)

# deduplicação baseada em timestamp_ingestion
window_spec = Window.partitionBy("id_produto").orderBy(F.col("timestamp_ingestion").desc())
df_produtos = df_produtos.withColumn("row_number", F.row_number().over(window_spec)) \
                         .filter(F.col("row_number") == 1) \
                         .drop("row_number")

# tratar categoria nula
df_produtos = df_produtos.withColumn(
    "categoria_produto",
    F.when(F.col("categoria_produto").isNull(), "Sem Categoria")
     .otherwise(F.col("categoria_produto"))
)

# criar schema se não existir e salvar na tabela final
spark.sql("create schema if not exists medallion.silver")
df_produtos.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.dim_produtos")

**Sellers**

In [0]:
from pyspark.sql.window import Window

df_sellers = spark.table("medallion.bronze.tb_sellers")

# renomear e cast
df_sellers = df_sellers.select(
    F.col("seller_id").cast("string").alias("id_vendedor"),
    F.col("seller_zip_code_prefix").cast("int").alias("prefixo_cep"),
    F.col("seller_city").cast("string").alias("cidade"),
    F.col("seller_state").cast("string").alias("estado"),
    F.col("seller_name").cast("string").alias("nome_vendedor"),
    F.to_date(F.col("seller_registration_date"), "yyyy-MM-dd").alias("data_registro_vendedor"),
    F.col("timestamp_ingestion")
)

# upper case em cidade e estado
df_sellers = df_sellers.withColumn("cidade", F.upper("cidade")) \
                       .withColumn("estado", F.upper("estado"))

# remover registros inválidos
df_sellers = df_sellers.filter(F.col("id_vendedor").isNotNull())

# deduplicação para não ter mais de 1 linha id_vendedor
window_spec = Window.partitionBy("id_vendedor").orderBy(F.col("timestamp_ingestion").desc())
df_sellers = df_sellers.withColumn("row_number", F.row_number().over(window_spec)) \
                       .filter(F.col("row_number") == 1) \
                       .drop("row_number")

# salvar na tabela final e criar schema se não existir
spark.sql("create schema if not exists medallion.silver")
df_sellers.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("medallion.silver.dim_vendedores")

Product Category Name

In [0]:
# eitura da tabela
df_pcn = spark.table("medallion.bronze.tb_product_category_name_translation")

# renomear e cast
df_pcn = df_pcn.select(
    F.col("product_category_name").cast("string").alias("nome_produto_pt"),
    F.col("product_category_name_english").cast("string").alias("nome_produto_en"),
    "timestamp_ingestion"
)

# removendo duplicatas
df_pcn = df_pcn.dropDuplicates()

# salvar na tabela final e verificar se schema existe
spark.sql("create schema if not exists silver")
df_pcn.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallion.silver.dim_categoria_produtos_traducao")

Cotação Dólar

In [0]:
# leitura da tabela e renomeio de colunas
df_dolar = spark.table("medallion.bronze.tb_cotacao_dolar")

# cast e renomear colunas
df_dolar = df_dolar.select(
    F.round(F.col("cotacaoCompra"), 2).cast("decimal(10,2)").alias("cotacao_dolar"),
    F.to_date(F.col("dataHoraCotacao"), "yyyy-MM-dd").alias("data_cotacao"),
    F.col("timestamp_ingestion")
)

# calcular min e max de data para criar calendário contínuo
datas = df_dolar.agg(
    F.min("data_cotacao").alias("min_data"),
    F.max("data_cotacao").alias("max_data")
).collect()[0]

min_data = datas["min_data"]
max_data = datas["max_data"]

# criar calendário contínuo de datas
df_calendario = spark.createDataFrame(
    [(min_data, max_data)],
    ["start", "end"]
).select(
    F.explode(F.sequence("start", "end")).alias("data_cotacao")
)

# join com cotação existente
df_join = df_calendario.join(
    df_dolar.select("data_cotacao", "cotacao_dolar"),
    on="data_cotacao",
    how="left"
)

# preencher cotação faltante com último valor disponível
window_spec = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
df_final = df_join.withColumn(
    "cotacao_dolar",
    F.last("cotacao_dolar", ignorenulls=True).over(window_spec)
).withColumn(
    "timestamp_ingestion",
    F.current_timestamp()
)

# salvar tabela silver
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medallion.silver.dim_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total Pedido

In [0]:
%sql
-- Cria a tabela final silver.fat_pedido_total
create or replace table medallion.silver.fat_pedido_total as

with pagamentos as (
    -- Agrega os pagamentos por pedido para obter o valor total pago em BRL
    select 
        id_pedido,
        sum(valor_pagamento) as valor_total_pago_brl
    from medallion.silver.fat_pagamentos_pedidos
    group by id_pedido
)

select
    p.id_pedido,
    p.id_consumidor,
    p.status_pedido as status,

    -- Valor total pago em BRL, arredondado para 2 casas decimais
    cast(round(pg.valor_total_pago_brl, 2) as decimal(10,2)) as valor_total_pago_brl,

    -- Valor total pago em USD, usando a cotação do dia da compra
    -- nullif evita divisão por zero caso não haja cotação
    cast(
        round(pg.valor_total_pago_brl / nullif(c.cotacao_dolar, 0), 2)
        as decimal(10,2)
    ) as valor_total_pago_usd,

    p.data_compra as data_pedido

from medallion.silver.fat_pedidos p

-- junta com pagamentos agregados
left join pagamentos pg
    on p.id_pedido = pg.id_pedido

-- junta com a cotação do dólar do dia da compra
left join medallion.silver.dim_cotacao_dolar c
    on to_date(p.data_compra) = c.data_cotacao
;

num_affected_rows,num_inserted_rows


Otimização tabelas fato

In [0]:
%sql
-- otimizando com zorder as colunas mais usadas em filtros e joins das tabelas fato da silver

-- fat_avaliacoes_pedidos: id_pedido e data_criacao_avaliacao
optimize medallion.silver.fat_avaliacoes_pedidos
zorder by (id_pedido, data_criacao_avaliacao);

-- fat_itens_pedidos: id_pedido e id_produto
optimize medallion.silver.fat_itens_pedidos
zorder by (id_pedido, id_produto);

-- fat_pagamentos_pedidos: id_pedido
optimize medallion.silver.fat_pagamentos_pedidos
zorder by (id_pedido);

-- fat_pedido_total: id_pedido e data_pedido
optimize medallion.silver.fat_pedido_total
zorder by (id_pedido, data_pedido);

-- fat_pedidos: id_pedido e data_compra
optimize medallion.silver.fat_pedidos
zorder by (id_pedido, data_compra);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 6618822), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1775229793793, 1775229794590, 8, 0, null, List(0, 0), null, 13, 13, 0, 0, null, null)"
